# 03b — Stock-specific factor selection

This step bridges factor analysis (notebook 03) and regression (notebook 04). It takes the factors accepted by notebook 03's `C_all` shortlist, then selects a separate ordered set for each stock. The market factor is always first. Additional factors are added greedily when they improve the training-sample BIC by at least 2, up to 8 total factors.

Both factor selection and orthogonalization are fitted only on **2017-01-03 through 2023-12-29**, matching the estimation period in notebooks 03 and 04. The saved projections are then applied unchanged to later dates. Use 2024 onward only for out-of-sample evaluation; the selection-window fit is not a performance estimate.

The main output is a stock-specific ordered factor list, saved projection coefficients, and a long table of orthogonalized factor returns. `regression_frame` assembles these into a date-indexed target-and-factor table for a later regression.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "src" / "factor_lens").exists():
    project_root = project_root.parent
if not (project_root / "src" / "factor_lens").exists():
    raise RuntimeError("Open this notebook from the repository root or a subfolder.")
sys.path.insert(0, str(project_root / "src"))

In [2]:
from factor_lens import cleaning, factor_correlation, factor_selection

clean = cleaning.get_clean()
correlation_results = factor_correlation.get_results()
selection_results = factor_selection.run_selection(
    correlation_results,
    clean["returns_excess"],
    candidate_set="C_all",
)
output_dir = factor_selection.save_results(selection_results, correlation_results)

print(f"Saved selection outputs to: {output_dir}")
print(f"Global candidates: {selection_results['metadata']['candidate_factors']}")
print(f"Stocks selected: {len(selection_results['selection'])}")
print(f"Stocks skipped: {len(selection_results['skipped'])}")

Cache hit: clean data already on Drive (config, raw data and overrides unchanged) -> loading.
Cache hit: factor results already on Drive for this clean data and config -> loading.
Saved selection outputs to: factor_lens_local\factors\factors_v1__clean_v1__yahoo_2016-01-01_2026-10-01_v1\stock_specific_selection_v1
Global candidates: ['LOWVOL', 'SEMI', 'SOFT', 'SIZE', 'COPPER', 'GRID', 'HY']
Stocks selected: 42
Stocks skipped: 5


## Per-stock results

Each row lists the factor order selected for that stock. `MKT` is mandatory; subsequent factors are orthogonalized against all earlier factors in that stock's order. BIC penalizes adding factors, and selection stops when no candidate reduces BIC by at least 2 or the factor cap is reached.

In [3]:
display(selection_results["selection"])
display(selection_results["skipped"])

,target,n_train,bic_market_only,bic_selected,factor_order,n_factors
0,AMD,1760,-12448.447211,-13025.519027,"[""MKT"", ""SEMI"", ""SOFT""]",3
1,AMZN,1760,-14518.822030,-15224.367821,"[""MKT"", ""SOFT"", ""LOWVOL"", ""SIZE"", ""HY"", ""GRID""...",7
2,ANET,1760,-13266.409042,-13415.833963,"[""MKT"", ""SOFT"", ""SEMI""]",3
3,APLD,430,-2045.038352,-2048.385103,"[""MKT"", ""LOWVOL""]",2
4,AVGO,1760,-14645.413353,-15238.831364,"[""MKT"", ""SEMI""]",2
5,BWXT,1760,-14606.331758,-14684.397761,"[""MKT"", ""SIZE"", ""LOWVOL""]",3
6,CAT,1760,-14751.176566,-15228.682281,"[""MKT"", ""COPPER"", ""SOFT"", ""SIZE"", ""LOWVOL""]",5
7,CCJ,1760,-12752.038844,-12910.779553,"[""MKT"", ""COPPER"", ""SIZE""]",3
8,CEG,489,-3737.073370,-3745.567769,"[""MKT"", ""LOWVOL"", ""COPPER""]",3
9,CLS,1760,-13341.058621,-13528.242447,"[""MKT"", ""SIZE"", ""GRID"", ""COPPER""]",4


,target,reason
0,ARM,only 74 complete training rows
1,CRWV,only 0 complete training rows
2,NBIS,only 0 complete training rows
3,TLN,only 145 complete training rows
4,GEV,only 0 complete training rows


## Prepare a regression input

Choose a stock below to get its excess-return target and already-orthogonalized factors. Fit the regression on the estimation dates, then assess it on the later holdout dates. Do not select or refit the factor projections using holdout outcomes.

In [4]:
target = selection_results["selection"].iloc[0]["target"]
regression_data = factor_selection.regression_frame(selection_results, target)
train_data = regression_data.loc[factor_selection.EST_START:factor_selection.EST_END]
holdout_data = regression_data.loc[
    (regression_data.index > factor_selection.EST_END)
    & (regression_data.index <= factor_selection.HOLD_END)
]

print(f"Target: {target}")
print(f"Selected factors: {list(train_data.columns[1:])}")
print(f"Training rows: {len(train_data)} | later evaluation rows: {len(holdout_data)}")
display(regression_data.head())

Target: AMD
Selected factors: ['MKT', 'SEMI', 'SOFT']
Training rows: 1760 | later evaluation rows: 690


,target_excess_return,MKT,SEMI,SOFT
date,,,,
2017-01-03,0.007917,0.007631,-0.009329,0.004040
2017-01-04,-0.000019,0.005930,-0.005436,0.008793
2017-01-05,-0.016644,-0.000815,-0.005108,0.004275
2017-01-06,0.007097,0.003558,-0.000177,0.005517
2017-01-09,0.014998,-0.003321,0.015745,-0.000973


The saved files are `stock_factor_selection.csv`, `orthogonalization_transforms.csv`, `orthogonalized_factors.parquet`, `skipped_targets.csv`, and `metadata.json`. The transform table records the training-fitted projection needed to reproduce each factor on later data.